In [6]:
import pandas as pd
import numpy as np

features = pd.read_csv("../data/customer_features_lowie1.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

test_df = test.merge(features, on="cust_id", how="left")

# number of customers with any missing values
print(test_df.isna().any(axis=1).sum())

# see which features have missing values and how many
print(test_df.isna().sum()[test_df.isna().sum() > 0])

test_df = test_df.fillna(0)

0
Series([], dtype: int64)


In [7]:
import joblib

feature_cols = joblib.load("../models/lowie1_feature_columns.pkl")
X_test = test_df[feature_cols]

In [ ]:
churn_lgb = joblib.load("../models/lowie1_churn_lgb_model.pkl")
churn_xgb = joblib.load("../models/lowie1_churn_xgb_model.pkl")
churn_cat = joblib.load("../models/lowie1_churn_cat_model.pkl")
churn_rf  = joblib.load("../models/lowie1_churn_rf_model.pkl")

iso_lgb = joblib.load("../models/lowie1_iso_lgb.pkl")
iso_xgb = joblib.load("../models/lowie1_iso_xgb.pkl")
iso_cat = joblib.load("../models/lowie1_iso_cat.pkl")
iso_rf  = joblib.load("../models/lowie1_iso_rf.pkl")

def predict_churn_proba(X):
    p_lgb = iso_lgb.transform(churn_lgb.predict_proba(X)[:, 1])
    p_xgb = iso_xgb.transform(churn_xgb.predict_proba(X)[:, 1])
    p_cat = iso_cat.transform(churn_cat.predict_proba(X)[:, 1])
    p_rf  = iso_rf.transform(churn_rf.predict_proba(X)[:, 1])
    return (p_lgb + p_xgb + p_cat + p_rf) / 4

In [9]:
p_return = predict_churn_proba(X_test)

In [10]:
log_rev_pred = revenue_model.predict(X_test)
rev_pred = np.expm1(log_rev_pred)
rev_pred = np.maximum(rev_pred, 0)

threshold = joblib.load("../models/lowie1_best_threshold.pkl")


final_pred = np.where(
    p_return < threshold,
    0,
    p_return * rev_pred
)

In [11]:
import os
os.makedirs("../submissions", exist_ok=True)

In [12]:
submission = pd.DataFrame({
    "cust_id": test_df["cust_id"],
    "prediction": final_pred
})

submission.to_csv("../submissions/submission_lowie1.csv", index=False)

In [2]:
import pandas as pd
sub = pd.read_csv("../submissions/submission_lowie1.csv")
print(sub.shape)
print(sub.head())
print(sub["prediction"].describe())
print(f"Zero predictions: {(sub['prediction'] == 0).sum()}")

(29148, 2)
            cust_id  prediction
0  2dfoualegmpt6x2h  352.709349
1  d2q2stjpnzld7a4r    0.000000
2  cojscuqlpylhclv2    0.000000
3  vntezlhi2ryvxk6m  146.798432
4  jgy4ytjkdr2b75wf  156.478953
count    29148.000000
mean        26.480049
std         67.124409
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        502.907149
Name: prediction, dtype: float64
Zero predictions: 24348
